# Praktikum Komputasi 2: Simulasi Transien Rangkaian RLC Seri (Julia)

Selamat datang di modul simulasi komputasi untuk **Persamaan Diferensial Orde 2** pada aplikasi rangkaian RLC seri.

## 📌 Pemodelan Rangkaian
Berdasarkan Hukum Tegangan Kirchhoff (KVL) pada rangkaian RLC seri tanpa sumber tegangan luar (respons alami / *zero-input response*):
$$ L \frac{d^2i}{dt^2} + R \frac{di}{dt} + \frac{1}{C} i = 0 $$

Dengan mendefinisikan koefisien redaman $\alpha$ dan frekuensi sudut natural $\omega_0$:
$$ \alpha = \frac{R}{2L}, \quad \omega_0 = \frac{1}{\sqrt{LC}} $$

Klasifikasi respons:
1. **Underdamped (Kurang Redam):** $\alpha < \omega_0$ (Arus berosilasi sambil meluruh secara eksponensial)
2. **Critically Damped (Redam Kritis):** $\alpha = \omega_0$ (Arus kembali ke nol paling cepat tanpa berosilasi)
3. **Overdamped (Lebih Redam):** $\alpha > \omega_0$ (Arus meluruh lambat menuju nol tanpa osilasi)

Di sini kita akan memecah PDB orde 2 menjadi sistem 2 PDB orde 1 dan menyelesaikannya secara numerik menggunakan Julia.

In [ ]:
using Plots

## 1. Reduksi Orde PDB ke Sistem Orde 1
Misalkan variabel keadaan (state variables):
$$ x_1 = i(t), \quad x_2 = \frac{di}{dt} $$

Sistem persamaan diferensial orde 1:
$$ \frac{dx_1}{dt} = x_2 $$
$$ \frac{dx_2}{dt} = -\frac{R}{L}x_2 - \frac{1}{LC}x_1 $$

In [ ]:
# Parameter Rangkaian
L = 1.0       # Induktansi (Henry)
C = 0.01      # Kapasitansi (Farad)
omega_0 = 1.0 / sqrt(L * C) # Frekuensi natural = 10 rad/s

# Nilai R untuk 3 skenario redaman
R_under = 4.0    # alpha = 2 < 10 (Underdamped)
R_crit  = 20.0   # alpha = 10 = 10 (Critically Damped)
R_over  = 50.0   # alpha = 25 > 10 (Overdamped)

println("Frekuensi sudut natural ω₀ = $omega_0 rad/s")
println("R Kritis = $(2 * L * omega_0) Ω")

## 2. Implementasi Numerik (Metode Runge-Kutta Orde 4 / RK4)
Metode RK4 memberikan akurasi yang jauh lebih tinggi daripada metode Euler untuk sistem osilasi orde 2.

In [ ]:
function solve_rlc_rk4(R, L, C, i0, didt0, t_max, dt)
    t_vec = 0.0:dt:t_max
    N = length(t_vec)
    
    # State: x[1] = i, x[2] = di/dt
    x = zeros(2, N)
    x[:, 1] = [i0, didt0]
    
    # Turunan: dx/dt = f(x)
    function f(state)
        i_val, didt_val = state[1], state[2]
        return [didt_val, -(R/L)*didt_val - (1.0/(L*C))*i_val]
    end
    
    for k in 1:(N-1)
        k1 = f(x[:, k])
        k2 = f(x[:, k] + 0.5 * dt * k1)
        k3 = f(x[:, k] + 0.5 * dt * k2)
        k4 = f(x[:, k] + dt * k3)
        
        x[:, k+1] = x[:, k] + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    end
    
    return t_vec, x[1, :]
end

# Simulasi 3 kondisi
t_max = 1.5
dt = 0.001
i0 = 0.0       # Arus awal 0 A
didt0 = 10.0   # Laju perubahan awal 10 A/s (akibat tegangan awal kapasitor)

t_arr, i_under = solve_rlc_rk4(R_under, L, C, i0, didt0, t_max, dt)
_,     i_crit  = solve_rlc_rk4(R_crit,  L, C, i0, didt0, t_max, dt)
_,     i_over  = solve_rlc_rk4(R_over,  L, C, i0, didt0, t_max, dt)

# Plot perbandingan ketiga respon
plot(t_arr, i_under, label="Underdamped (R = 4 Ω)", lw=2.5, color=:blue, xlabel="Waktu (detik)", ylabel="Arus i(t) [Ampere]")
plot!(t_arr, i_crit,  label="Critically Damped (R = 20 Ω)", lw=2.5, color=:green)
plot!(t_arr, i_over,  label="Overdamped (R = 50 Ω)", lw=2.5, color=:red)
hline!([0], color=:black, ls=:dash, label="")
title!("Perbandingan Respons Transien RLC Seri (Julia RK4)")

## 3. Penyelesaian Menggunakan SciML (DifferentialEquations.jl)
Mari kita selesaikan kasus yang sama menggunakan pustaka standar industri `DifferentialEquations.jl`.

In [ ]:
using DifferentialEquations

# Definisikan PDB dalam format SciML: f(du, u, p, t)
# u[1] = i, u[2] = di/dt
# p = [R, L, C]
function rlc_ode!(du, u, p, t)
    R, L, C = p
    du[1] = u[2]
    du[2] = -(R/L)*u[2] - (1.0/(L*C))*u[1]
end

u0 = [0.0, 10.0]
tspan = (0.0, 1.5)
params = [R_under, L, C]

prob = ODEProblem(rlc_ode!, u0, tspan, params)
sol = solve(prob, Tsit5(), reltol=1e-6, abstol=1e-6)

# Plot arus i(t)
plot(sol, vars=(0, 1), label="Arus i(t) - SciML Tsit5", lw=2, color=:purple, xlabel="Waktu (detik)", ylabel="Arus (A)")
title!("Solusi Adaptif RLC Seri dengan Tsit5()")

## 4. Tugas Eksplorasi Mahasiswa
1. **Analisis Frekuensi:** Pada kasus *underdamped*, hitung secara analitik frekuensi osilasi teredam $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$. Cocokkan periodenya ($T = 2\pi / \omega_d$) dengan jarak puncak ke puncak pada grafik!
2. **PDB Non-Homogen:** Ubahlah fungsi diferensial dengan menambahkan tegangan generator sinusoidal $V(t) = 10 \sin(10t)$. Amati fenomena resonansi yang terjadi ketika frekuensi sumber mendekati frekuensi natural $\omega_0$!